# EdgeYOLO Dataset Builder (BDD100K Day/Night)

Dieses Notebook erstellt `exports/edgeyolodataset` mit den geforderten Splits,
kopiert zufaellig ausgewaehlte Bilder, generiert Train_Night via CycleGAN
und schreibt YOLO-TXT Labels nur fuer die verwendeten Bilder.


## Ablauf

1. Zufalls-Sampling der Quellbilder pro Split und Kopieren nach `exports/edgeyolodataset/<Split>/images`.
2. Train_Night aus Train_Day via CycleGAN (day2night) erzeugen und auf Originalgroesse zurueckskalieren.
3. BDD100K JSON-Labels nur fuer die verwendeten Bilder in YOLO-TXT konvertieren.
4. Fehlende Labels protokollieren.


In [ ]:
from pathlib import Path
import json
import random
import shutil
from typing import Dict, List, Sequence, Tuple

import torch
from PIL import Image

import os

os.chdir("/srv/store/docker-users/thesis/khajuria/day2night")
print("Current working directory:", os.getcwd())

from src.apply_cyclegan import (
    SUPPORTED_EXTENSIONS,
    build_generator,
    build_inference_transform,
    find_checkpoint,
    gather_image_paths,
    tensor_to_pil,
)
from src.utils.common import load_cfg


In [ ]:
exports_root = Path("exports/edgeyolodataset_label_bdd")

train_day_src = Path("data/bdd_kaggle_labels/bdd100k/bdd100k/images/100k/train/trainA")
train_night_real_src = Path("data/bdd_kaggle_labels/bdd100k/bdd100k/images/100k/train/trainB")
val_day_src = Path("data/bdd_kaggle_labels/bdd100k/bdd100k/images/100k/train/trainA")
val_night_src = train_night_real_src
test_night_src = Path("data/bdd_kaggle_labels/bdd100k/bdd100k/images/100k/train/testB")
test_day_src = Path("data/bdd_kaggle_labels/bdd100k/bdd100k/images/100k/train/testA")

train_labels_json = Path("data/bdd_kaggle_labels/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json")
val_labels_json = train_labels_json

cyclegan_config = Path("experiments/lastchance_bdd_samgan_20260615_030739/semgan_bdd.yaml")
checkpoint_path = Path("experiments/lastchance_bdd_samgan_20260615_030739/latest.pt")

random_seed = 43

split_specs = {
    "Train_Day": {"src": train_day_src, "count": 3000, "labels_json": train_labels_json},
    "Train_Night_Real": {"src": train_night_real_src, "count": 3000, "labels_json": train_labels_json},
    "Val_day": {"src": val_day_src, "count": 300, "labels_json": val_labels_json},
    "Val_night": {"src": val_night_src, "count": 300, "labels_json": val_labels_json},
    "Test_Night": {"src": test_night_src, "count": 1000, "labels_json": train_labels_json},
    "Test_Day": {"src": test_day_src, "count": 1000, "labels_json": train_labels_json},
}


In [ ]:
BDD_YOLO_CLASSES = [
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motor",
    "bike",
    "traffic light",
    "traffic sign",
]

# Trage hier Klassen ein, die nicht in den YOLO-Labels landen sollen.
# Beispiel: EXCLUDED_YOLO_CLASSES = {"traffic light", "traffic sign"}
EXCLUDED_YOLO_CLASSES = {"bus", "bike", "rider", "motor", "train"}

unknown_excluded_classes = EXCLUDED_YOLO_CLASSES - set(BDD_YOLO_CLASSES)
if unknown_excluded_classes:
    raise ValueError(f"Unknown excluded classes: {sorted(unknown_excluded_classes)}")

YOLO_CLASSES = [name for name in BDD_YOLO_CLASSES if name not in EXCLUDED_YOLO_CLASSES]
CLASS_TO_ID = {name: idx for idx, name in enumerate(YOLO_CLASSES)}
print(f"YOLO classes ({len(YOLO_CLASSES)}): {YOLO_CLASSES}")
if EXCLUDED_YOLO_CLASSES:
    print(f"Excluded classes: {sorted(EXCLUDED_YOLO_CLASSES)}")


In [ ]:
def sample_images(src_dir: Path, count: int, extensions: Sequence[str]) -> List[Path]:
    images = gather_image_paths(src_dir, extensions)
    if count > len(images):
        raise ValueError(f"Not enough images in {src_dir} ({len(images)} available, {count} requested).")
    return random.sample(images, count)


def sample_images_excluding(
    src_dir: Path,
    count: int,
    extensions: Sequence[str],
    exclude_names: set,
) -> List[Path]:
    images = [p for p in gather_image_paths(src_dir, extensions) if p.name not in exclude_names]
    if count > len(images):
        raise ValueError(f"Not enough images in {src_dir} after exclusion ({len(images)} available, {count} requested).")
    return random.sample(images, count)


def copy_images(image_paths: Sequence[Path], src_root: Path, dest_images_dir: Path) -> List[Path]:
    dest_images_dir.mkdir(parents=True, exist_ok=True)
    dest_paths: List[Path] = []
    for image_path in image_paths:
        rel_path = image_path.relative_to(src_root)
        dest_path = dest_images_dir / rel_path
        dest_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(image_path, dest_path)
        dest_paths.append(dest_path)
    return dest_paths


def load_bdd_labels(json_path: Path, image_names: set) -> Dict[str, List[dict]]:
    with json_path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    labels_index: Dict[str, List[dict]] = {}
    for entry in data:
        name = entry.get("name")
        if name in image_names:
            labels_index[name] = entry.get("labels", [])
    return labels_index


def clamp(value: float, min_value: float, max_value: float) -> float:
    return max(min_value, min(value, max_value))


def bbox_to_yolo(box2d: dict, img_w: int, img_h: int) -> Tuple[float, float, float, float]:
    x1 = clamp(float(box2d["x1"]), 0.0, float(img_w))
    y1 = clamp(float(box2d["y1"]), 0.0, float(img_h))
    x2 = clamp(float(box2d["x2"]), 0.0, float(img_w))
    y2 = clamp(float(box2d["y2"]), 0.0, float(img_h))
    width = max(0.0, x2 - x1)
    height = max(0.0, y2 - y1)
    x_center = x1 + width / 2.0
    y_center = y1 + height / 2.0
    return (x_center / img_w, y_center / img_h, width / img_w, height / img_h)


def write_yolo_labels(
    image_paths: Sequence[Path],
    labels_index: Dict[str, List[dict]],
    labels_dir: Path,
    class_to_id: Dict[str, int],
) -> List[str]:
    labels_dir.mkdir(parents=True, exist_ok=True)
    missing: List[str] = []
    for image_path in image_paths:
        name = image_path.name
        labels = labels_index.get(name)
        if labels is None:
            missing.append(name)
            continue
        with Image.open(image_path) as img:
            img_w, img_h = img.size
        yolo_lines: List[str] = []
        for label in labels:
            category = label.get("category")
            if category not in class_to_id:
                continue
            box2d = label.get("box2d")
            if not box2d:
                continue
            x_center, y_center, box_w, box_h = bbox_to_yolo(box2d, img_w, img_h)
            yolo_lines.append(
                f"{class_to_id[category]} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}"
            )
        label_path = labels_dir / f"{image_path.stem}.txt"
        label_path.write_text("\n".join(yolo_lines))
    return missing


def write_missing_report(missing_by_split: Dict[str, List[str]], output_path: Path) -> None:
    lines: List[str] = []
    for split_name, names in missing_by_split.items():
        lines.append(f"[{split_name}]")
        for name in sorted(set(names)):
            lines.append(name)
        lines.append("")
    output_path.write_text("\n".join(lines))


def generate_train_night(
    image_paths: Sequence[Path],
    input_root: Path,
    output_dir: Path,
    config_path: Path,
    checkpoint_path: str | None,
    device: torch.device,
) -> List[Path]:
    if checkpoint_path is None:
        raise ValueError("Set checkpoint_path to a CycleGAN checkpoint .pt file.")
    cfg = load_cfg(config_path)
    cfg.setdefault("transforms", {})["resize"] = [1024, 1024]
    cfg["transforms"]["center_crop"] = None
    cfg["transforms"]["random_crop"] = None
    checkpoint = find_checkpoint(checkpoint_path, cfg)
    transform = build_inference_transform(cfg)
    inference_cfg = cfg.get("inference", {})
    brightness_gain = float(inference_cfg.get("brightness_gain", 1.0))
    output_gamma = float(inference_cfg.get("output_gamma", 1.0))
    generator = build_generator(cfg, device, "day2night")
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    generator.load_state_dict(state["G"])
    generator.eval()

    output_dir.mkdir(parents=True, exist_ok=True)
    output_paths: List[Path] = []
    with torch.inference_mode():
        for idx, image_path in enumerate(image_paths, start=1):
            img = Image.open(image_path).convert("RGB")
            orig_size = img.size
            input_tensor = transform(img).unsqueeze(0).to(device)
            output_tensor = generator(input_tensor)
            result_img = tensor_to_pil(
                output_tensor, brightness_gain=brightness_gain, output_gamma=output_gamma
            )
            if result_img.size != orig_size:
                result_img = result_img.resize(orig_size, resample=Image.BICUBIC)
            rel_path = image_path.relative_to(input_root)
            dest_path = output_dir / rel_path
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            result_img.save(dest_path)
            output_paths.append(dest_path)
            if idx % 50 == 0:
                print(f"Converted {idx}/{len(image_paths)} images", end="\r")
    print(f"\nSaved {len(output_paths)} images to {output_dir}")
    return output_paths


In [ ]:
random.seed(random_seed)
extensions = [ext.lower().lstrip(".") for ext in SUPPORTED_EXTENSIONS]

split_outputs = {}
train_day_names = set()
train_night_real_names = set()
for split_name, spec in split_specs.items():
    if split_name == "Val_day":
        samples = sample_images_excluding(spec["src"], spec["count"], extensions, train_day_names)
    elif split_name == "Val_night":
        samples = sample_images_excluding(spec["src"], spec["count"], extensions, train_night_real_names)
    else:
        samples = sample_images(spec["src"], spec["count"], extensions)
    if split_name == "Train_Day":
        train_day_names.update(p.name for p in samples)
    elif split_name == "Train_Night_Real":
        train_night_real_names.update(p.name for p in samples)
    images_dir = exports_root / split_name / "images"
    labels_dir = exports_root / split_name / "labels"
    export_paths = copy_images(samples, spec["src"], images_dir)
    split_outputs[split_name] = {
        "samples": samples,
        "export_paths": export_paths,
        "labels_dir": labels_dir,
        "labels_json": spec["labels_json"],
    }
    print(f"{split_name}: {len(export_paths)} images copied to {images_dir}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
train_night_images_dir = exports_root / "Train_Night" / "images"
train_night_labels_dir = exports_root / "Train_Night" / "labels"

train_night_export_paths = generate_train_night(
    image_paths=split_outputs["Train_Day"]["samples"],
    input_root=train_day_src,
    output_dir=train_night_images_dir,
    config_path=cyclegan_config,
    checkpoint_path=checkpoint_path,
    device=device,
)
print(f"Train_Night images: {len(train_night_export_paths)}")


In [ ]:
train_image_names = set()
for split_name in ("Train_Day", "Train_Night_Real", "Test_Night", "Test_Day"):
    train_image_names.update(p.name for p in split_outputs[split_name]["export_paths"])
train_labels_index = load_bdd_labels(train_labels_json, train_image_names)

val_image_names = set()
for split_name in ("Val_day", "Val_night"):
    val_image_names.update(p.name for p in split_outputs[split_name]["export_paths"])
val_labels_index = load_bdd_labels(val_labels_json, val_image_names)

missing: Dict[str, List[str]] = {}
missing["Train_Day"] = write_yolo_labels(
    split_outputs["Train_Day"]["export_paths"],
    train_labels_index,
    split_outputs["Train_Day"]["labels_dir"],
    CLASS_TO_ID,
)

missing["Train_Night_Real"] = write_yolo_labels(
    split_outputs["Train_Night_Real"]["export_paths"],
    train_labels_index,
    split_outputs["Train_Night_Real"]["labels_dir"],
    CLASS_TO_ID,
)
missing["Val_day"] = write_yolo_labels(
    split_outputs["Val_day"]["export_paths"],
    val_labels_index,
    split_outputs["Val_day"]["labels_dir"],
    CLASS_TO_ID,
)
missing["Val_night"] = write_yolo_labels(
    split_outputs["Val_night"]["export_paths"],
    val_labels_index,
    split_outputs["Val_night"]["labels_dir"],
    CLASS_TO_ID,
)
missing["Test_Night"] = write_yolo_labels(
    split_outputs["Test_Night"]["export_paths"],
    train_labels_index,
    split_outputs["Test_Night"]["labels_dir"],
    CLASS_TO_ID,
)
missing["Test_Day"] = write_yolo_labels(
    split_outputs["Test_Day"]["export_paths"],
    train_labels_index,
    split_outputs["Test_Day"]["labels_dir"],
    CLASS_TO_ID,
)

train_night_labels_dir.mkdir(parents=True, exist_ok=True)
for label_path in split_outputs["Train_Day"]["labels_dir"].glob("*.txt"):
    shutil.copy2(label_path, train_night_labels_dir / label_path.name)
missing["Train_Night"] = list(missing["Train_Day"])

missing_report_path = exports_root / "missing_labels.txt"
write_missing_report(missing, missing_report_path)
print(f"Missing labels written to: {missing_report_path}")


### Hinweise

- Setze `checkpoint_path` vor der Ausfuehrung des Train_Night-Schritts.
- Die fehlenden Label-Namen stehen in `exports/edgeyolodataset/missing_labels.txt`.


### Ersatzsuche fuer fehlende Labels

Dieser Abschnitt versucht, fehlende Labels aus `missing_labels.txt`
in den vorhandenen BDD100K JSON-Dateien zu finden und zu schreiben.


In [ ]:
missing_report_path = exports_root / "missing_labels.txt"
if not missing_report_path.exists():
    raise FileNotFoundError(f"Missing report not found: {missing_report_path}")

missing_by_split = {}
current_split = None
for raw in missing_report_path.read_text().splitlines():
    line = raw.strip()
    if not line:
        continue
    if line.startswith("[") and line.endswith("]"):
        current_split = line[1:-1]
        missing_by_split[current_split] = []
        continue
    if current_split:
        missing_by_split[current_split].append(line)

extensions = [ext.lower().lstrip(".") for ext in SUPPORTED_EXTENSIONS]
replacement_missing = {}

train_day_dir = exports_root / "Train_Day" / "images"
train_day_names = {p.name for p in train_day_dir.rglob("*") if p.is_file()}
train_night_real_dir = exports_root / "Train_Night_Real" / "images"
train_night_real_names = {p.name for p in train_night_real_dir.rglob("*") if p.is_file()}

def remove_missing_export_files(images_dir: Path, labels_dir: Path, names: Sequence[str]) -> int:
    removed = 0
    for name in set(names):
        stem = Path(name).stem
        candidates = list(images_dir.rglob(name)) + list(labels_dir.rglob(f"{stem}.txt"))
        for path in candidates:
            if path.is_file():
                path.unlink()
                removed += 1
    return removed

for split_name, names in missing_by_split.items():
    needed = len(set(names))
    if needed == 0:
        replacement_missing[split_name] = []
        continue
    spec = split_specs.get(split_name)
    if spec is None:
        print(f"Skipping unknown split in report: {split_name}")
        continue
    src_dir = spec["src"]
    labels_json = spec["labels_json"]
    images_dir = exports_root / split_name / "images"
    labels_dir = exports_root / split_name / "labels"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    removed = remove_missing_export_files(images_dir, labels_dir, names)
    if split_name == "Train_Day":
        removed += remove_missing_export_files(
            exports_root / "Train_Night" / "images",
            exports_root / "Train_Night" / "labels",
            names,
        )
    print(f"{split_name}: removed {removed} old files before replacement")

    existing_names = {p.name for p in images_dir.rglob("*") if p.is_file()}
    if split_name == "Val_day":
        existing_names.update(train_day_names)
    elif split_name == "Val_night":
        existing_names.update(train_night_real_names)

    samples = sample_images_excluding(src_dir, needed, extensions, existing_names)
    export_paths = copy_images(samples, src_dir, images_dir)

    labels_index = load_bdd_labels(labels_json, {p.name for p in export_paths})
    replacement_missing[split_name] = write_yolo_labels(
        export_paths,
        labels_index,
        labels_dir,
        CLASS_TO_ID,
    )

    if split_name == "Train_Day":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        train_night_images_dir = exports_root / "Train_Night" / "images"
        train_night_labels_dir = exports_root / "Train_Night" / "labels"
        train_night_export_paths = generate_train_night(
            image_paths=samples,
            input_root=train_day_src,
            output_dir=train_night_images_dir,
            config_path=cyclegan_config,
            checkpoint_path=checkpoint_path,
            device=device,
        )
        train_night_labels_dir.mkdir(parents=True, exist_ok=True)
        new_label_names = {p.stem for p in export_paths}
        for label_path in labels_dir.glob("*.txt"):
            if label_path.stem in new_label_names:
                shutil.copy2(label_path, train_night_labels_dir / label_path.name)
        print(f"Train_Night replacements: {len(train_night_export_paths)}")

replacement_report_path = exports_root / "missing_labels_replacement.txt"
write_missing_report(replacement_missing, replacement_report_path)

print("Replacement run complete.")
for split_name, names in replacement_missing.items():
    print(f"  {split_name}: {len(names)} missing after replacement")
print(f"Replacement missing labels written to: {replacement_report_path}")


### Kombinierte Splits

Dieser Abschnitt erstellt kombinierte Ordner als Symlink-Sammlungen:

- `Val_Combined`: `Val_day` und `Val_night`
- `Train_Combined_B`: `Train_Day` und `Train_Night`
- `Train_Combined_C`: `Train_Day` und `Train_Night_Real`


In [ ]:
combined_specs = {
    "Val_Combined": ["Val_day", "Val_night"],
    "Train_Combined_B": ["Train_Day", "Train_Night"],
    "Train_Combined_C": ["Train_Day", "Train_Night_Real"],
}


def recreate_symlink(link_path: Path, target_path: Path) -> None:
    if link_path.exists() or link_path.is_symlink():
        if link_path.is_symlink():
            link_path.unlink()
        else:
            raise FileExistsError(f"Refusing to replace non-symlink path: {link_path}")
    link_path.symlink_to(target_path, target_is_directory=True)


for combined_name, split_names in combined_specs.items():
    combined_dir = exports_root / combined_name
    combined_dir.mkdir(parents=True, exist_ok=True)
    for split_name in split_names:
        split_dir = exports_root / split_name
        if not split_dir.exists():
            raise FileNotFoundError(f"Required split folder missing: {split_dir}")
        recreate_symlink(combined_dir / split_name, Path("..") / split_name)
    print(f"{combined_name}: linked {', '.join(split_names)}")


### Train_Combined_D

Dieser Abschnitt kopiert eine gemischte Trainingsmenge in eine flache YOLO-Struktur:

- 1500 aus `Train_Night`
- 1500 aus `Train_Night_Real`
- 3000 aus `Train_Day`

Die Dateien bekommen einen Split-Prefix, damit gleiche Dateinamen aus `Train_Day` und `Train_Night` nicht kollidieren.


In [ ]:
train_combined_d_specs = {
    "Train_Night": 1500,
    "Train_Night_Real": 1500,
    "Train_Day": 3000,
}
train_combined_d_dir = exports_root / "Train_Combined_D"
train_combined_d_images_dir = train_combined_d_dir / "images"
train_combined_d_labels_dir = train_combined_d_dir / "labels"


def clear_directory_contents(directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    for path in directory.iterdir():
        if path.is_dir() and not path.is_symlink():
            shutil.rmtree(path)
        else:
            path.unlink()


def prefixed_export_stem(split_name: str, image_path: Path, images_dir: Path) -> str:
    rel = image_path.relative_to(images_dir).with_suffix("")
    rel_stem = "__".join(rel.parts)
    return f"{split_name}__{rel_stem}"


def copy_yolo_subset_to_combined(
    split_name: str,
    count: int,
    dest_images_dir: Path,
    dest_labels_dir: Path,
    rng: random.Random,
) -> int:
    images_dir = exports_root / split_name / "images"
    labels_dir = exports_root / split_name / "labels"
    if not images_dir.exists():
        raise FileNotFoundError(f"Missing images directory: {images_dir}")
    if not labels_dir.exists():
        raise FileNotFoundError(f"Missing labels directory: {labels_dir}")

    image_paths = [
        p
        for p in images_dir.rglob("*")
        if p.is_file() and p.suffix.lower().lstrip(".") in extensions
    ]
    if count > len(image_paths):
        raise ValueError(f"Not enough images in {images_dir}: {len(image_paths)} available, {count} requested")

    selected = rng.sample(sorted(image_paths), count)
    for image_path in selected:
        new_stem = prefixed_export_stem(split_name, image_path, images_dir)
        dest_image_path = dest_images_dir / f"{new_stem}{image_path.suffix.lower()}"
        shutil.copy2(image_path, dest_image_path)

        src_label_path = labels_dir / f"{image_path.stem}.txt"
        if not src_label_path.exists():
            raise FileNotFoundError(f"Missing label for {image_path}: {src_label_path}")
        shutil.copy2(src_label_path, dest_labels_dir / f"{new_stem}.txt")
    return len(selected)


clear_directory_contents(train_combined_d_images_dir)
clear_directory_contents(train_combined_d_labels_dir)

combined_rng = random.Random(random_seed)
combined_d_counts = {}
for split_name, count in train_combined_d_specs.items():
    combined_d_counts[split_name] = copy_yolo_subset_to_combined(
        split_name,
        count,
        train_combined_d_images_dir,
        train_combined_d_labels_dir,
        combined_rng,
    )

print(f"Train_Combined_D written to {train_combined_d_dir}")
for split_name, count in combined_d_counts.items():
    print(f"  {split_name}: {count}")
print(f"  total: {sum(combined_d_counts.values())}")
